<a href="https://colab.research.google.com/github/sonos4849-a11y/task-1/blob/main/bonus_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Automated Weekly Churn Prediction Pipeline

To create an automated weekly churn prediction pipeline, we need to encapsulate our model and preprocessing steps into reusable functions. This pipeline would typically involve:

1.  **Loading new raw data:** Simulating the weekly arrival of new customer data.
2.  **Preprocessing:** Applying the exact same feature engineering and scaling steps used during training.
3.  **Prediction:** Loading the trained model and generating churn probabilities for the new data.
4.  **Risk Categorization:** Assigning 'Low', 'Medium', or 'High' risk categories based on the predicted probabilities.
5.  **Output and Reporting:** Saving the predictions and possibly generating reports or alerts.
6.  **Scheduling:** Automating the execution of this pipeline weekly (this part is conceptual for Colab but crucial in a real-world setup).

Let's start by saving our trained XGBoost model and then creating a preprocessing function.

### 1. Save the Trained XGBoost Model

To use our model in an automated pipeline, we first need to save it to disk. We'll use `joblib` for this purpose, as it's efficient for scikit-learn compatible models.

In [ ]:
import joblib
import os

# Define a directory to save models
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

# Save the XGBoost model
model_filename = os.path.join(model_dir, 'xgboost_churn_model.joblib')
joblib.dump(xgb_model, model_filename)

print(f"XGBoost model saved to: {model_filename}")

XGBoost model saved to: models/xgboost_churn_model.joblib


### 2. Create a Data Preprocessing Function

Consistency in preprocessing is vital for any prediction pipeline. We'll create a function that takes raw customer data (similar to our initial `df`) and applies all the feature engineering and scaling steps we performed earlier. This function will also need access to the `StandardScaler` used during training, so we'll save that too.

In [ ]:
# Save the StandardScaler fitted on the training data
scaler_filename = os.path.join(model_dir, 'scaler.joblib')
joblib.dump(scaler, scaler_filename)

print(f"StandardScaler saved to: {scaler_filename}")

def preprocess_new_data(raw_data_df, trained_scaler):
    # Make a copy to avoid modifying the original DataFrame
    processed_df = raw_data_df.copy()

    # Drop customerID if it exists
    if 'customerID' in processed_df.columns:
        processed_df = processed_df.drop(["customerID"], axis=1)

    # Convert 'TotalCharges' to numeric and handle missing values
    processed_df['TotalCharges'] = pd.to_numeric(processed_df['TotalCharges'], errors='coerce')
    processed_df.dropna(subset=['TotalCharges'], inplace=True)

    # Bin 'tenure' into categories
    # Ensure bins are consistent with training (adjusted for max tenure)
    bins = [0, 12, 24, 48, 60, 73] # Max tenure observed is 72, so upper bound is 73
    labels = ['0-12 Mths', '12-24 Mths', '24-48 Mths', '48-60 Mths', '60+ Mths']
    processed_df['tenure_group'] = pd.cut(processed_df['tenure'], bins=bins, labels=labels, right=False, include_lowest=True)

    # Create interaction features: MonthlyCharges_ContractType
    processed_df['MonthlyCharges_MonthToMonth'] = processed_df['MonthlyCharges'] * (processed_df['Contract'] == 'Month-to-month').astype(int)
    processed_df['MonthlyCharges_OneYear'] = processed_df['MonthlyCharges'] * (processed_df['Contract'] == 'One year').astype(int)
    processed_df['MonthlyCharges_TwoYear'] = processed_df['MonthlyCharges'] * (processed_df['Contract'] == 'Two year').astype(int)

    # Convert 'No phone service' and 'No internet service' to 'No' for consistency
    for col in ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
        if col in processed_df.columns:
            processed_df[col] = processed_df[col].replace({'No phone service': 'No', 'No internet service': 'No'})

    # List of binary columns to encode (should match training) for mapping 0/1
    binary_cols_for_mapping = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
        'PaperlessBilling'
    ] # Churn is removed as it's the target

    for col in binary_cols_for_mapping:
        if col in processed_df.columns:
            if col == 'gender':
                processed_df[col] = processed_df[col].map({'Female': 0, 'Male': 1})
            else:
                processed_df[col] = processed_df[col].map({'No': 0, 'Yes': 1})

    # Create 'Total Services' Feature (after binary encoding of services)
    processed_df['TotalServices'] = processed_df[[
        'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]].sum(axis=1)
    # 'InternetService' needs to be handled carefully as it's categorical for one-hot encoding later
    # For TotalServices, we just need to know if they have internet service (DSL/Fiber Optic) or not ('No')
    processed_df['TotalServices'] += (processed_df['InternetService'] != 'No').astype(int)

    # One-Hot Encoding Remaining Categorical Features
    categorical_cols_for_ohe = ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group']
    processed_df = pd.get_dummies(processed_df, columns=categorical_cols_for_ohe, drop_first=True)

    # Ensure all columns from training are present and in the same order, fill missing with 0
    # This is critical if new data is missing a category that was present in training
    training_columns = X.columns # X was the feature set used for training
    missing_cols = set(training_columns) - set(processed_df.columns)
    for c in missing_cols:
        processed_df[c] = 0
    processed_df = processed_df[training_columns]

    # Apply scaling to numerical features (using the *trained* scaler)
    numerical_cols_for_scaling = processed_df.select_dtypes(include=np.number).columns.tolist()
    processed_df[numerical_cols_for_scaling] = trained_scaler.transform(processed_df[numerical_cols_for_scaling])

    return processed_df

print("Preprocessing function 'preprocess_new_data' defined.")

StandardScaler saved to: models/scaler.joblib
Preprocessing function 'preprocess_new_data' defined.


### 3. Create a Prediction Function for New Data

This function will integrate the loading of the model and scaler, apply the preprocessing function, and then make predictions, including assigning risk categories.

In [ ]:
def make_predictions_and_assign_risk(new_raw_data_df, model_path, scaler_path, low_thresh=0.3, medium_thresh=0.7):
    # Load the trained model and scaler
    loaded_model = joblib.load(model_path)
    loaded_scaler = joblib.load(scaler_path)

    # Preprocess the new raw data
    processed_new_data = preprocess_new_data(new_raw_data_df.copy(), loaded_scaler)

    # Generate churn probabilities
    new_predictions_proba = loaded_model.predict_proba(processed_new_data)[:, 1]

    # Create a DataFrame for results
    prediction_results = pd.DataFrame({
        'Original_Index': new_raw_data_df.index, # Keep original index for reference
        'Predicted_Churn_Probability': new_predictions_proba
    })

    # Assign risk categories
    def assign_risk_category_pipeline(prob):
        if prob < low_thresh:
            return 'Low Risk'
        elif prob < medium_thresh:
            return 'Medium Risk'
        else:
            return 'High Risk'

    prediction_results['Risk_Category'] = prediction_results['Predicted_Churn_Probability'].apply(assign_risk_category_pipeline)

    return prediction_results

print("Prediction function 'make_predictions_and_assign_risk' defined.")

Prediction function 'make_predictions_and_assign_risk' defined.


### 4. Simulate Weekly Prediction

To demonstrate the pipeline, let's simulate the arrival of new weekly data. We'll take a small sample from our original `data` DataFrame, pretending it's new data, and pass it through our `make_predictions_and_assign_risk` function.

In [ ]:
# Simulate new raw data (e.g., a batch of customers from the original dataset)
# For a real pipeline, this would be new data loaded from a database or file
simulated_new_raw_data = data.sample(n=20, random_state=100) # Taking 20 random samples

print("Simulated new raw data:")
display(simulated_new_raw_data.head())

# Run the prediction pipeline on the simulated new data
weekly_churn_predictions = make_predictions_and_assign_risk(
    simulated_new_raw_data,
    model_filename, # Path to the saved model
    scaler_filename  # Path to the saved scaler
)

print("\nWeekly churn predictions for simulated new data:")
display(weekly_churn_predictions.head(10))

Simulated new raw data:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4880,1215-EXRMO,Male,0,Yes,No,50,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.55,1067.65,No
1541,2429-AYKKO,Male,0,No,No,72,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.85,1434.1,No
1289,9968-FFVVH,Male,0,No,No,63,Yes,Yes,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),68.80,4111.35,No
5745,3580-GICBM,Female,0,Yes,Yes,61,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,24.20,1445.2,No
4873,2320-YKQBO,Female,0,No,No,7,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.30,144.95,No



Weekly churn predictions for simulated new data:


,Original_Index,Predicted_Churn_Probability,Risk_Category
0,4880,0.006246,Low Risk
1,1541,0.000205,Low Risk
2,1289,0.010472,Low Risk
3,5745,0.002540,Low Risk
4,4873,0.252797,Low Risk
5,4168,0.001512,Low Risk
6,1557,0.098155,Low Risk
7,2892,0.087038,Low Risk
8,664,0.020540,Low Risk
9,1588,0.525751,Medium Risk


In [ ]:
df.duplicates(inplace=True)
print("Duplicate rows removed. New DataFrame shape:", df.shape)

Duplicate rows removed. New DataFrame shape: (7021, 20)


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            0 non-null      float64
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           0 non-null      float64
 3   Dependents        0 non-null      float64
 4   tenure            7032 non-null   int64  
 5   PhoneService      0 non-null      float64
 6   MultipleLines     0 non-null      float64
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    0 non-null      float64
 9   OnlineBackup      0 non-null      float64
 10  DeviceProtection  0 non-null      float64
 11  TechSupport       0 non-null      float64
 12  StreamingTV       0 non-null      float64
 13  StreamingMovies   0 non-null      float64
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  0 non-null      float64
 16  PaymentMethod     7032 non-null   object 
 17  